# Hotel Booking Cancellation & Overbooking Decision
## 05 — Feature Engineering, Train/Test Split and Preprocessing

Prepare the cleaned data for ML without fitting preprocessing on the test set.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.project_utils import (
    RANDOM_STATE,
    TARGET,
    BOOKING_TIME_EXCLUSIONS,
    prepare_model_data,
    stratified_train_test_split,
    build_preprocessors,
)

print("Repository root:", REPO_ROOT)
print("Random state:", RANDOM_STATE)


## 2. Load Cleaned Data

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Processed dataset not found. Run Notebook 03 first: "
        "03_data_cleaning.ipynb"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Cleaned shape:", df.shape)
df.head()


## 3. Confirm Leakage Columns Are Absent

In [ ]:
direct_leakage = [
    "reservation_status",
    "reservation_status_date",
]

remaining_direct_leakage = [
    c for c in direct_leakage
    if c in df.columns
]

print("Direct outcome leakage remaining:", remaining_direct_leakage)

assert not remaining_direct_leakage


## 4. Canonical Booking-Time Feature Engineering

The reusable transformation is implemented in `src/project_utils.py` so the modelling notebooks cannot silently drift apart.

At this stage the workflow:

- excludes `assigned_room_type`, `booking_changes`, and `days_in_waiting_list` from booking-time prediction;
- creates `total_nights`;
- creates `total_guests`;
- creates `total_previous_bookings`;
- creates `has_special_request`;
- removes redundant source columns after their information has been consolidated.

Notebook 03 remains responsible for raw-data cleaning and direct outcome-leakage removal.


In [ ]:
df_model = prepare_model_data(df)

print(
    "Booking-time exclusions:",
    BOOKING_TIME_EXCLUSIONS
)

print("Model-ready shape:", df_model.shape)

print(
    "Booking-time exclusions remaining:",
    [
        c for c in BOOKING_TIME_EXCLUSIONS
        if c in df_model.columns
    ]
)

df_model[
    [
        "total_nights",
        "total_guests",
        "total_previous_bookings",
        "has_special_request",
    ]
].head()


## 5. Remove Redundant Source Columns

In [ ]:
redundant_source_columns = [
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
]

remaining_redundant = [
    c for c in redundant_source_columns
    if c in df_model.columns
]

print("Redundant source columns remaining:", remaining_redundant)
print("Model-ready shape:", df_model.shape)

assert not remaining_redundant


## 6. Define X and y

In [ ]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

print("X:", X.shape)
print("y:", y.shape)

print("\nTarget distribution (%):")
print(
    (y.value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

assert TARGET not in X.columns


## 7. Canonical Stratified Train/Test Split

The project uses one shared 80/20 stratified split with `random_state=42`. Centralising the split prevents training, evaluation, and threshold notebooks from accidentally evaluating different observations.


In [ ]:
X_train, X_test, y_train, y_test = stratified_train_test_split(
    X,
    y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print(
    "Train cancellation %:",
    round(y_train.mean() * 100, 2),
)
print(
    "Test cancellation %:",
    round(y_test.mean() * 100, 2),
)


## 8. Shared Preprocessing Policy

Feature types and preprocessors are created by the shared project utility.

Numerical features use median imputation and standardisation in the standard model pipeline. Categorical features use most-frequent imputation and one-hot encoding. Rare `country` levels below 1% of the training data are learned from the training fold and grouped into `Other`.

The boosting pipeline uses ordinal encoding rather than one-hot encoding while retaining the same training-fitted rare-country policy.


In [ ]:
(
    preprocessor,
    boost_preprocessor,
    numerical_features,
    categorical_features,
) = build_preprocessors(X_train)

print(
    "Numerical:",
    len(numerical_features),
    numerical_features,
)

print(
    "\nCategorical:",
    len(categorical_features),
    categorical_features,
)

print(
    "\nCountry policy: categories below 1% "
    "training frequency are grouped into 'Other'."
)


## 9. Numerical Pipeline

Median imputation plus standardization.

In [ ]:
numeric_pipeline = preprocessor.named_transformers_["num"] if hasattr(
    preprocessor, "transformers_"
) else preprocessor.transformers[0][1]

print(numeric_pipeline)


## 10. Categorical Pipeline

Most-frequent imputation plus one-hot encoding. Unknown test categories are ignored.

In [ ]:
categorical_pipeline = preprocessor.named_transformers_["cat"] if hasattr(
    preprocessor, "transformers_"
) else preprocessor.transformers[1][1]

print(categorical_pipeline)


## 11. Combined Preprocessor

In [ ]:
print("Shared standard preprocessor:")
print(preprocessor)

print("\nShared boosting preprocessor:")
print(boost_preprocessor)


## 12. Fit Only on Training Data

This is a critical leakage-control step. The preprocessor is fitted only on `X_train`; the held-out test set is transformed using the already-fitted training transformation.


In [ ]:
# Fit preprocessing on training data only.
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already-fitted transformation to held-out test data.
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed test:", X_test_processed.shape)


## 13. Generated Feature Names

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))
print(feature_names[:30])


## 14. Validation

In [ ]:
checks = {
    "Train rows preserved":
        X_train_processed.shape[0] == X_train.shape[0],

    "Test rows preserved":
        X_test_processed.shape[0] == X_test.shape[0],

    "Same processed columns":
        X_train_processed.shape[1] == X_test_processed.shape[1],

    "Target excluded":
        TARGET not in X.columns,

    "No direct leakage":
        not any(
            c in X.columns
            for c in [
                "reservation_status",
                "reservation_status_date",
            ]
        ),

    "No booking-time exclusions":
        not any(
            c in X.columns
            for c in BOOKING_TIME_EXCLUSIONS
        ),
}

for label, passed in checks.items():
    print(f"{label}: {passed}")

assert all(checks.values())

print("\nPASS: preprocessing validation")


## Modelling Note

Notebook 05 demonstrates the shared feature and preprocessing contract. Notebooks 06–08 import the same reusable implementation from `src/project_utils.py`, ensuring that model comparison, held-out evaluation, and cost-sensitive threshold selection operate on the same booking-time feature definition.


## Summary

Completed feature engineering, stratified train/test splitting, numerical/categorical identification, one-hot encoding and scaling pipelines, and leakage-safe preprocessing verification.

### Next Step
Train Logistic Regression, KNN, Decision Tree, Random Forest and a boosting model, then compare Accuracy, Precision, Recall, F1, ROC-AUC and confusion matrices.